(lesson-19)=
# 3.19 Benthic Semantic Segmentation for Reef Cover

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ada-carter/cv/blob/main/book/BenthicSemSeg_SegFormer.ipynb)

## Overview

Reef ecologists need to know what covers the seafloor: coral, algae, sand, or rock. The standard field method is a photo-transect survey, where a camera moves along a fixed line and the analyst scores fixed points in each frame. Automating that scoring with pixel-level predictions converts a bottleneck task into a scalable pipeline.

This lesson uses **semantic segmentation**, which assigns one class label to every pixel in an image. That is different from instance segmentation (lesson 3.10), which identifies and outlines individual objects. Semantic segmentation classifies regions: every coral pixel gets label 0, every algae pixel gets label 1, and so on. The output is a mask the same size as the input image. Summing up pixels by class and dividing by the total gives **percent cover**, the primary metric in reef monitoring surveys.

The model is **SegFormer** with the `mit-b0` backbone. SegFormer is a transformer-based segmentation architecture that uses a hierarchical Mix Transformer (MiT) encoder and a lightweight MLP decoder. The `mit-b0` variant has roughly 3.7 million parameters, which is small enough to fine-tune on Colab's free GPU tier in a few minutes.

### Learning Objectives

By the end of this lesson you will be able to:

- Explain the difference between semantic and instance segmentation and when each is appropriate for reef monitoring tasks.
- Load a pretrained SegFormer model from HuggingFace Transformers and configure it for a custom number of benthic classes.
- Fine-tune SegFormer on labeled benthic imagery, including handling the 1/4-resolution output that SegFormer produces.
- Compute per-class percent cover statistics from a predicted segmentation mask.
- Visualize color-coded cover maps overlaid on the original image for qualitative assessment.

In [ ]:
!pip install -q transformers datasets torch torchvision pillow matplotlib numpy
!nvidia-smi

## Dataset

We use a small benthic image dataset modeled on the annotation format used by **NOAA CoralNet**, the platform most commonly used for reef survey data management. In CoralNet, each image is annotated at the pixel level (or point level for sparse surveys). For semantic segmentation training, we need dense pixel-level masks where each pixel carries an integer class label.

The five classes in this lesson are:

| Index | Class | Description |
|-------|-------|-------------|
| 0 | coral | Hard and soft coral tissue |
| 1 | algae | Macroalgae and turf algae |
| 2 | sand | Bare sand and rubble |
| 3 | rock | Exposed hard substrate |
| 4 | fish_other | Fish and unclassified foreground objects |

The code below attempts to load a small labeled dataset from HuggingFace (`OceanCV/BenthicSeg`). If that dataset is not yet available, it falls back to generating 10 synthetic image/mask pairs with random pixel values. The synthetic data lets you step through the full training and inference pipeline without any real imagery. When you run this on real reef data the only change is the dataset source.

In [ ]:
from datasets import load_dataset
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

CLASS_NAMES = ["coral", "algae", "sand", "rock", "fish_other"]
CLASS_COLORS = [
    (255, 127, 80),   # coral: coral orange
    (34, 139, 34),    # algae: forest green
    (210, 180, 140),  # sand: tan
    (105, 105, 105),  # rock: grey
    (70, 130, 180),   # fish/other: steel blue
]

try:
    dataset = load_dataset("OceanCV/BenthicSeg", split="train")
    print(f"Loaded {len(dataset)} images from HuggingFace")
except Exception:
    print("Dataset not found - generating synthetic demo data")
    # Generate 10 synthetic image/mask pairs
    images = [Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)) for _ in range(10)]
    masks = [np.random.randint(0, 5, (256, 256), dtype=np.uint8) for _ in range(10)]
    dataset = [{"image": img, "mask": mask} for img, mask in zip(images, masks)]
    print(f"Generated {len(dataset)} synthetic samples for demonstration")

## Visualizing Annotations

Before training on any dataset, inspect a sample of image/mask pairs. This step catches common problems: swapped class indices, all-zero masks due to a loading bug, or annotations that do not align with the image content. Overlaying the color-coded mask on the original image at 60/40 transparency is a fast visual check that confirms both spatial alignment and class coverage.

In [ ]:
def colorize_mask(mask_array):
    """Convert integer class mask to RGB color image."""
    h, w = mask_array.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for class_idx, color in enumerate(CLASS_COLORS):
        rgb[mask_array == class_idx] = color
    return rgb

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row, sample in enumerate(dataset[:3]):
    img = np.array(sample["image"].convert("RGB") if hasattr(sample["image"], "convert") else sample["image"])
    mask = np.array(sample["mask"]) if not isinstance(sample["mask"], np.ndarray) else sample["mask"]
    overlay = (img * 0.6 + colorize_mask(mask) * 0.4).astype(np.uint8)
    axes[row, 0].imshow(img); axes[row, 0].set_title("Original"); axes[row, 0].axis("off")
    axes[row, 1].imshow(colorize_mask(mask)); axes[row, 1].set_title("Mask"); axes[row, 1].axis("off")
    axes[row, 2].imshow(overlay); axes[row, 2].set_title("Overlay"); axes[row, 2].axis("off")
plt.tight_layout()
plt.show()

## Loading SegFormer

SegFormer uses a hierarchical Mix Transformer (MiT) encoder that produces feature maps at four different scales, from 1/4 to 1/32 of the input resolution. Unlike earlier vision transformers, MiT does not use fixed positional embeddings, which means it generalizes better to images of different sizes at inference time. The decoder is a small all-MLP head that concatenates upsampled features from all four encoder stages and produces the final class logits at 1/4 of input resolution. You then upsample those logits to the original image size using bilinear interpolation to compute loss and generate predictions.

`mit-b0` is the smallest variant in the MiT family. It has 3.7M parameters, trains quickly, and is appropriate for datasets under a few thousand images.

We load the model from HuggingFace using `SegformerForSemanticSegmentation.from_pretrained`. Setting `num_labels=5` and `ignore_mismatched_sizes=True` replaces the pretrained classification head (trained on ADE20K's 150 classes) with a fresh head sized for our 5 benthic classes. The encoder weights are preserved and fine-tuned from their pretrained state.

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

id2label = {i: name for i, name in enumerate(CLASS_NAMES)}
label2id = {name: i for i, name in enumerate(CLASS_NAMES)}

processor = SegformerImageProcessor.from_pretrained(
    "nvidia/mit-b0",
    do_reduce_labels=False
)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model = model.to(device)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

## Fine-tuning

The training loop below runs for 3 epochs, which is enough to demonstrate that the loss decreases. On a real dataset you would train for 50 to 100 epochs with a larger batch size and a learning rate schedule.

A few details worth noting:

**Upsampling the logits.** SegFormer outputs logits at 1/4 of the input resolution. The ground truth masks are at full resolution. Before computing cross-entropy loss, we upsample the logits with `F.interpolate(..., mode="bilinear", align_corners=False)` to match the mask size. Alternatively, you can downsample the masks, but upsampling the logits is standard practice and preserves spatial precision.

**Optimizer.** We use a single AdamW optimizer over all parameters at `lr=6e-5`. In a full training setup you would typically use a lower learning rate for the pretrained encoder and a higher rate for the fresh decoder head, since the decoder starts from random weights. That differential LR strategy converges faster and avoids overwriting useful pretrained features.

**Loss.** `CrossEntropyLoss(ignore_index=255)` treats pixels labeled 255 as unlabeled and excludes them from the gradient. This is the standard convention for partially annotated masks.

In [ ]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

class BenthicDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = sample["image"].convert("RGB") if hasattr(sample["image"], "convert") else Image.fromarray(sample["image"])
        mask = np.array(sample["mask"])
        encoding = self.processor(images=image, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.long)
        return pixel_values, mask_tensor

train_ds = BenthicDataset(dataset, processor)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)

optimizer = AdamW(model.parameters(), lr=6e-5)
loss_fn = CrossEntropyLoss(ignore_index=255)

model.train()
for epoch in range(3):
    total_loss = 0
    for pixel_values, masks in train_loader:
        pixel_values = pixel_values.to(device)
        masks = masks.to(device)
        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits
        # Upsample to mask size
        upsampled = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        loss = loss_fn(upsampled, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3 - Loss: {total_loss/len(train_loader):.4f}")
print("Training complete.")

## Inference and Percent Cover

After training, we run the model on a single test image. The model outputs class logits for each pixel. Taking the argmax across the class dimension gives a single predicted class index per pixel, which is the predicted segmentation mask.

**Percent cover** is then straightforward: count the number of pixels belonging to each class and divide by the total number of pixels. The result is a number between 0 and 100 for each class, representing the proportion of the image covered by that benthic category. In an operational reef monitoring pipeline, you would run this calculation over every frame in a video transect and average the results, or produce a spatial map of cover variation along the transect line.

This single-image percent cover estimate is analogous to the point-count score that a human analyst would produce from the same image using CoralNet, but it uses all pixels rather than a sparse grid of scored points.

In [ ]:
model.eval()
test_sample = dataset[0]
test_image = test_sample["image"].convert("RGB") if hasattr(test_sample["image"], "convert") else Image.fromarray(test_sample["image"])

with torch.no_grad():
    inputs = processor(images=test_image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    logits = outputs.logits
    upsampled = F.interpolate(logits, size=(256, 256), mode="bilinear", align_corners=False)
    pred_mask = upsampled.argmax(dim=1).squeeze().cpu().numpy()

# Compute percent cover
total_pixels = pred_mask.size
print("\nPercent Cover Estimates:")
for i, class_name in enumerate(CLASS_NAMES):
    count = (pred_mask == i).sum()
    pct = 100 * count / total_pixels
    print(f"  {class_name}: {pct:.1f}%")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_image); axes[0].set_title("Input Image"); axes[0].axis("off")
axes[1].imshow(colorize_mask(pred_mask)); axes[1].set_title("Predicted Mask"); axes[1].axis("off")
overlay = (np.array(test_image) * 0.6 + colorize_mask(pred_mask) * 0.4).astype(np.uint8)
axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")

# Legend
from matplotlib.patches import Patch
legend = [Patch(color=[c/255 for c in col], label=name) for name, col in zip(CLASS_NAMES, CLASS_COLORS)]
axes[1].legend(handles=legend, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## Reflecting on Results

Work through these questions before moving on.

**1. Pixel-based percent cover vs. point-count surveys.**
Traditional reef surveys use point-count methods: a diver or analyst scores the benthic category at a fixed grid of points in each photo (commonly 50 points per image in CoralNet). Semantic segmentation scores every pixel. How do the two approaches compare in terms of precision, coverage, and potential bias? Are there habitats or imaging conditions where point-count might give more reliable estimates than a segmentation model trained on limited data?

**2. Class imbalance during fine-tuning.**
Many reef environments are dominated by sand or rock, so those classes may account for 80 or 90 percent of pixels in a training set. How would this imbalance affect model predictions? Describe at least two strategies you could use during training to mitigate it: one based on the loss function and one based on data sampling or augmentation.

**3. Extending to video transects.**
A video transect records several minutes of continuous seafloor footage as a diver or ROV moves along a fixed line. How would you adapt this inference pipeline to process the full video and produce a time-series or distance-series percent cover curve for each class? Consider frame sampling rate, temporal smoothing, and how you would synchronize percent cover estimates with the physical distance traveled along the transect.